# Fast SAMSum summaries with a fine-tuned model

This notebook skips training and uses `philschmid/flan-t5-base-samsum`, a public FLAN-T5 checkpoint already fine-tuned on SAMSum. Its model weights are about 945 MB, so the first run needs to download them.

In Google Colab, choose **Runtime > Change runtime type > T4 GPU** for faster generation, then run the cells in order. No dataset download, Drive mount, or training run is needed. You can edit the dialogue in the final cell to try your own conversation.

In [ ]:
# Install only the Transformers and tokenizer libraries needed for inference.
%pip install -q "transformers>=4.46,<5" "sentencepiece>=0.2,<1"

In [ ]:
# Import PyTorch and the model/tokenizer loaders used for text generation.
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# Select the public FLAN-T5 checkpoint already fine-tuned on SAMSum.
MODEL_ID = "philschmid/flan-t5-base-samsum"

# Load its vocabulary and trained summarization weights from Hugging Face.
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)

# Use Colab's GPU when available; inference can also run on CPU, more slowly.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()
print(f"Loaded {MODEL_ID} on {device}")

In [ ]:
# Generate a summary from one dialogue without running any training.
def summarize(dialogue: str) -> str:
    # Reject empty input so the model is not asked to summarize blank text.
    if not dialogue.strip():
        raise ValueError("Enter a non-empty dialogue to summarize.")

    # Convert dialogue text to model input tokens and keep input within context.
    inputs = tokenizer(
        dialogue.strip(),
        return_tensors="pt",
        truncation=True,
        max_length=512,
    ).to(device)

    # Generate without gradients because the model is being used only for inference.
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=96,
            num_beams=4,
            no_repeat_ngram_size=3,
        )

    # Decode tokens and normalize any visible Pegasus-style newline markers.
    output = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return " ".join(output.replace("<n>", " ").split())

In [ ]:
# Replace this sample with your own speaker-labelled conversation.
dialogue = """Doctor: What brings you here today?

Patient: I've had a headache and felt unusually tired for the last three days.

Doctor: Have you experienced a fever or vomiting?

Patient: No fever, but I haven't been sleeping well."""

# Print the generated summary in a readable form.
print(summarize(dialogue))